# Notebook 2 - Chess: Implementing Pieces, Board, Move and Game

In Notebook 1 we designed the classes. Now let's **build** them and watch the design pay off.

**Plan**

1. `Piece` hierarchy - King, Queen, Rook, Bishop, Knight, Pawn.
2. `Board` - a thin 8x8 container with pretty-printing.
3. `Move` - a tiny data class so we can **undo** moves.
4. `Game` - turn order, legal-move checks, **basic check detection**, move history.
5. A runnable mini-game plus common real-world rules (pawn's first double step, captures).

> Everything runs. Each cell builds on the previous one - run them in order.


## Setup

```bash
cd 07-object-oriented-design/chess
uv sync
```

In VS Code pick the `.venv` kernel (top-right). If it doesn't appear, reload the window
(`Cmd+Shift+P` then **Reload Window**).


## 1. The `Piece` hierarchy

All six pieces in one cell so it's easy to scan. Note how **each class owns its own rules**.
Also notice the shared `slide` helper - Rook, Bishop, and Queen are all "sliding pieces",
so we write the ray-casting logic **once**.


In [1]:
from abc import ABC, abstractmethod

WHITE, BLACK = 'W', 'B'

def on_board(r, c):
    return 0 <= r < 8 and 0 <= c < 8


class Piece(ABC):
    def __init__(self, color):
        self.color = color

    @abstractmethod
    def symbol(self) -> str: ...

    @abstractmethod
    def valid_moves(self, board, r, c): ...

    def __repr__(self):
        s = self.symbol()
        return s.upper() if self.color == WHITE else s.lower()


# ---- Helper for sliding pieces (Rook, Bishop, Queen) ----
def slide(board, r, c, color, directions):
    out = []
    for dr, dc in directions:
        nr, nc = r+dr, c+dc
        while on_board(nr, nc):
            if board[nr][nc] is None:
                out.append((nr, nc))
            else:
                if board[nr][nc].color != color:
                    out.append((nr, nc))  # capture
                break
            nr += dr; nc += dc
    return out


class King(Piece):
    def symbol(self): return 'k'
    def valid_moves(self, board, r, c):
        out = []
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                if dr == 0 and dc == 0:
                    continue
                nr, nc = r+dr, c+dc
                if on_board(nr, nc) and (board[nr][nc] is None or board[nr][nc].color != self.color):
                    out.append((nr, nc))
        return out


class Rook(Piece):
    def symbol(self): return 'r'
    def valid_moves(self, board, r, c):
        return slide(board, r, c, self.color, [(-1,0),(1,0),(0,-1),(0,1)])


class Bishop(Piece):
    def symbol(self): return 'b'
    def valid_moves(self, board, r, c):
        return slide(board, r, c, self.color, [(-1,-1),(-1,1),(1,-1),(1,1)])


class Queen(Piece):
    """Queen = Rook + Bishop. Literally."""
    def symbol(self): return 'q'
    def valid_moves(self, board, r, c):
        return slide(board, r, c, self.color,
                     [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)])


class Knight(Piece):
    def symbol(self): return 'n'
    def valid_moves(self, board, r, c):
        deltas = [(-2,-1),(-2,1),(-1,-2),(-1,2),(1,-2),(1,2),(2,-1),(2,1)]
        out = []
        for dr, dc in deltas:
            nr, nc = r+dr, c+dc
            if on_board(nr, nc) and (board[nr][nc] is None or board[nr][nc].color != self.color):
                out.append((nr, nc))
        return out


class Pawn(Piece):
    """Pawns are weird: forward 1, forward 2 on first move, capture diagonally."""
    def symbol(self): return 'p'
    def valid_moves(self, board, r, c):
        dr = -1 if self.color == WHITE else 1   # white moves toward row 0
        start_row = 6 if self.color == WHITE else 1
        out = []
        # forward 1
        if on_board(r+dr, c) and board[r+dr][c] is None:
            out.append((r+dr, c))
            # forward 2 from starting row
            if r == start_row and board[r+2*dr][c] is None:
                out.append((r+2*dr, c))
        # diagonal captures
        for dc in (-1, 1):
            nr, nc = r+dr, c+dc
            if on_board(nr, nc) and board[nr][nc] and board[nr][nc].color != self.color:
                out.append((nr, nc))
        return out

print('Pieces defined:', [c.__name__ for c in (King, Queen, Rook, Bishop, Knight, Pawn)])


Pieces defined: ['King', 'Queen', 'Rook', 'Bishop', 'Knight', 'Pawn']


## 2. The `Board` - a thin container

The board does **not** know chess rules. It only:

- holds an 8x8 grid,
- sets up the starting position,
- prints itself.

That's the Single Responsibility Principle: one reason to change.


In [2]:
class Board:
    FILES = 'abcdefgh'

    def __init__(self):
        self.grid = [[None]*8 for _ in range(8)]

    def setup_start(self):
        back = [Rook, Knight, Bishop, Queen, King, Bishop, Knight, Rook]
        for c, cls in enumerate(back):
            self.grid[0][c] = cls(BLACK)
            self.grid[7][c] = cls(WHITE)
        for c in range(8):
            self.grid[1][c] = Pawn(BLACK)
            self.grid[6][c] = Pawn(WHITE)
        return self

    def __getitem__(self, rc):
        r, c = rc
        return self.grid[r][c]

    def __setitem__(self, rc, value):
        r, c = rc
        self.grid[r][c] = value

    def show(self):
        print('   ' + ' '.join(self.FILES))
        for r, row in enumerate(self.grid):
            rank = 8 - r
            line = ' '.join(str(p) if p else '.' for p in row)
            print(f'{rank}  {line}  {rank}')
        print('   ' + ' '.join(self.FILES))

b = Board().setup_start()
b.show()


   a b c d e f g h
8  r n b q k b n r  8
7  p p p p p p p p  7
6  . . . . . . . .  6
5  . . . . . . . .  5
4  . . . . . . . .  4
3  . . . . . . . .  3
2  P P P P P P P P  2
1  R N B Q K B N R  1
   a b c d e f g h


## 3. The `Move` data class - so we can undo

A `Move` is just a record: *from where, to where, and what (if anything) was captured*.
Holding onto the captured piece lets us **undo** - which is exactly how chess engines
explore "what if I play this?" without actually committing.


In [3]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class Move:
    frm: tuple            # (row, col)
    to: tuple             # (row, col)
    piece: Piece          # what moved
    captured: Optional[Piece] = None   # what was taken, if anything

    def __repr__(self):
        files = 'abcdefgh'
        def sq(rc):
            r, c = rc
            return f'{files[c]}{8 - r}'
        cap = 'x' if self.captured else '-'
        return f'{self.piece}{sq(self.frm)}{cap}{sq(self.to)}'

m = Move(frm=(6,4), to=(4,4), piece=Pawn(WHITE))
print('Example move:', m)


Example move: Pe2-e4


## 4. The `Game` - turn order, making moves, undo, check detection

Now we combine everything. The `Game`:

- keeps whose turn it is,
- validates a move by asking the piece,
- records moves in history,
- supports **undo**,
- detects a simple **"is the king in check?"** condition.

> **Check detection intuition:** the king is "in check" if *any* enemy piece's valid moves
> include the king's square. We already have `valid_moves` on every piece - reuse it!


In [4]:
class Game:
    def __init__(self, board=None):
        self.board = board or Board().setup_start()
        self.turn = WHITE
        self.history: list = []

    # ----- locating the king (needed for check detection) -----
    def _find_king(self, color):
        for r in range(8):
            for c in range(8):
                p = self.board[r, c]
                if isinstance(p, King) and p.color == color:
                    return (r, c)
        return None

    def in_check(self, color):
        king_pos = self._find_king(color)
        if king_pos is None:
            return False
        enemy = BLACK if color == WHITE else WHITE
        for r in range(8):
            for c in range(8):
                p = self.board[r, c]
                if p and p.color == enemy:
                    if king_pos in p.valid_moves(self.board.grid, r, c):
                        return True
        return False

    # ----- making and undoing moves -----
    def move(self, frm, to):
        r, c = frm; nr, nc = to
        p = self.board[r, c]
        if p is None:
            raise ValueError(f'no piece at {frm}')
        if p.color != self.turn:
            raise ValueError(f"it is {self.turn}'s turn, not {p.color}'s")
        if (nr, nc) not in p.valid_moves(self.board.grid, r, c):
            raise ValueError(f'illegal move for {p}: {frm} -> {to}')

        captured = self.board[nr, nc]
        self.board[nr, nc] = p
        self.board[r, c] = None

        # Rule: a move cannot leave YOUR OWN king in check.
        if self.in_check(self.turn):
            self.board[r, c] = p
            self.board[nr, nc] = captured
            raise ValueError('illegal: would leave your king in check')

        self.history.append(Move(frm, to, p, captured))
        self.turn = BLACK if self.turn == WHITE else WHITE

    def undo(self):
        if not self.history:
            raise ValueError('nothing to undo')
        m = self.history.pop()
        r, c = m.frm; nr, nc = m.to
        self.board[r, c] = m.piece
        self.board[nr, nc] = m.captured
        self.turn = m.piece.color   # back to the mover's turn

print('Game class ready.')


Game class ready.


## 5. Play a tiny opening

Let's play a few moves, print the board, then check and undo.

> **Coordinate reminder:** our grid uses `(row, col)` with row 0 at the top (Black's side).
> So White's `e2` pawn is at `(6, 4)` and moving to `e4` means `(4, 4)`.


In [5]:
g = Game()
print('Starting position:')
g.board.show()

# A famous opening: 1. e4 e5  2. Nf3 Nc6
g.move((6,4), (4,4))   # white:  e2 -> e4
g.move((1,4), (3,4))   # black:  e7 -> e5
g.move((7,6), (5,5))   # white:  Ng1 -> f3
g.move((0,1), (2,2))   # black:  Nb8 -> c6

print()
print('After 4 moves:')
g.board.show()
print('Move history:')
for m in g.history:
    print(' ', m)
print('White in check?', g.in_check(WHITE))


Starting position:
   a b c d e f g h
8  r n b q k b n r  8
7  p p p p p p p p  7
6  . . . . . . . .  6
5  . . . . . . . .  5
4  . . . . . . . .  4
3  . . . . . . . .  3
2  P P P P P P P P  2
1  R N B Q K B N R  1
   a b c d e f g h

After 4 moves:
   a b c d e f g h
8  r . b q k b n r  8
7  p p p p . p p p  7
6  . . n . . . . .  6
5  . . . . p . . .  5
4  . . . . P . . .  4
3  . . . . . N . .  3
2  P P P P . P P P  2
1  R N B Q K B . R  1
   a b c d e f g h
Move history:
  Pe2-e4
  pe7-e5
  Ng1-f3
  nb8-c6
White in check? False


### Undo

In [6]:
g.undo()
g.undo()
print('After undoing 2 moves:')
g.board.show()
print('Moves remaining in history:', len(g.history))
print('Whose turn:', g.turn)


After undoing 2 moves:
   a b c d e f g h
8  r n b q k b n r  8
7  p p p p . p p p  7
6  . . . . . . . .  6
5  . . . . p . . .  5
4  . . . . P . . .  4
3  . . . . . . . .  3
2  P P P P . P P P  2
1  R N B Q K B N R  1
   a b c d e f g h
Moves remaining in history: 2
Whose turn: W


### Illegal moves raise clear errors

Great error messages are part of good OOD: the user (or tests) should know *why* a move failed.


In [7]:
def try_move(g, frm, to, label):
    try:
        g.move(frm, to)
        print(f'{label}: OK')
    except ValueError as e:
        print(f'{label}: rejected -> {e}')

try_move(g, (7,1), (6,0), 'knight to non-L square')   # illegal shape
try_move(g, (1,0), (2,0), 'moving opponent piece')    # wrong color
try_move(g, (6,4), (3,4), 'pawn jumping too far')     # pawn cannot leap 3


knight to non-L square: rejected -> illegal move for N: (7, 1) -> (6, 0)
moving opponent piece: rejected -> it is W's turn, not B's
pawn jumping too far: rejected -> no piece at (6, 4)


## 6. A tiny "in check" demo

Let's construct a position where Black's king is attacked by a White rook,
and confirm `in_check` returns `True`.


In [8]:
demo = Board()
demo.grid[7][4] = King(WHITE)   # white king on e1
demo.grid[0][4] = King(BLACK)   # black king on e8
demo.grid[4][4] = Rook(WHITE)   # rook on e4, same file as black king
demo.show()

g2 = Game(demo)
g2.turn = BLACK   # it's black's turn to respond to check
print('Black in check?', g2.in_check(BLACK))
print('White in check?', g2.in_check(WHITE))


   a b c d e f g h
8  . . . . k . . .  8
7  . . . . . . . .  7
6  . . . . . . . .  6
5  . . . . . . . .  5
4  . . . . R . . .  4
3  . . . . . . . .  3
2  . . . . . . . .  2
1  . . . . K . . .  1
   a b c d e f g h
Black in check? True
White in check? False


## 7. What we still skipped (and why)

These are great **exercises** for you to extend:

- **Castling** - needs "have I moved yet?" flags on King/Rook and path checks.
- **En-passant** - needs to know the *previous* move (we already have `history`!).
- **Pawn promotion** - when a pawn reaches the last rank, swap it for a Queen.
- **Checkmate / stalemate** - "no legal move and in check" / "no legal move and not in check".
- **Draw by repetition / 50-move rule** - needs a position hash, which is what chess engines do.

Notice how each of these slots cleanly into our existing classes. That's the win: the
design we built in Notebook 1 can *grow* without a rewrite.


## OOD takeaways

1. **Polymorphism beats `if/elif` chains.** Each piece knows its own rules.
2. **Separate concerns.** `Board` stores, `Piece` decides moves, `Game` enforces order.
3. **Model actions as objects.** `Move` makes undo and history trivial.
4. **Reuse knowledge.** `in_check` reuses `valid_moves` - no duplicate logic.
5. **Leave room to grow.** Castling, en-passant, promotion all fit without breaking the shape.

If you remember only one thing: **put the knowledge where it naturally lives.**
